# Paper 1 — Descarga y diagnostico v6  (GFW + WDPA) — version auditada

Esta version fue revisada celda por celda contra los datos reales. Lo que se corrigio:

| # | problema | consecuencia | arreglo |
|---|---|---|---|
| 1 | el filtro `MARINE`/`STATUS` **nunca funciono** (v2→v5 siempre dio 272 de 272) | se grillaban parques terrestres de fiordo | filtro robusto en cascada + imprime `value_counts` + **lista completa de nombres** para que la revises |
| 2 | la celda 8 buffereaba el poligono **crudo** (100k vertices) para la bbox | se colgaba igual que la 7 | bbox por expansion de bounds, instantanea |
| 3 | `drop_duplicates(WDPAID)` botaba parcelas de una misma AMP | perdida de area | `dissolve` por WDPAID (union de parcelas) |
| 4 | resolucion `HIGH` (0,01°) para una grilla de 0,1° | **100x** mas datos de los necesarios | `LOW` (0,1°), que anida exacto en 0,1° y 0,2° |
| 5 | oceanicas a 0,25° no anidan en la grilla GFW de 0,1° | efecto de borde en los bins | oceanicas a **0,2°** |
| 6 | panel denso `celdas x 84 meses` en RAM y a CSV | 25M filas, GBs, minutos | diagnostico sobre panel **sparse**; el denso se arma despues, por caso |
| 7 | distancia al borde punto-a-poligono sin indice | ~10⁹ ops en fiordos | segmentos + `STRtree.nearest` |
| 8 | `base_reach` con loop Python por celda | lento con 300k celdas | vectorizado |
| 9 | efecto en el binning de efecto de AMPs con distinto delta que se solapan | atribucion arbitraria | **la celda mas fina gana**, sin doble conteo |
| 10 | sin reintentos ante 429/5xx | un hilo fallaba y habia que re-correr | 4 intentos con backoff |
| 11 | `no_take='Part'` tratado igual que `'All'` | sobreestima infraccion | se conserva `no_take_kind` para distinguir |
| 12 | `STATUS='designated'` solo | excluia Inscribed/Established (vigentes) | acepta las tres, excluye Proposed |
| 13 | registro GFW justo en el borde de un bin | podia caer al bin vecino por float | `BIN_EPS` hacia el interior de la celda |
| 14 | el anillo de AMPs costeras cubria **tierra** | inflaba el denominador de h/celda | mascara de tierra Natural Earth 50m |
| 15 | se botaba el arte de pesca | feature util perdida | `h_trawl/h_longline/h_squid/h_seine/h_other` |

Y una **celda de autotest** (6b) que verifica dedup, binning, alcance y distancia **antes** de tocar la API.

**Que me mandas:** `DIAGNOSTIC.txt` y `SCHEMA_SAMPLE.txt`.

---
### Corre las celdas de arriba a abajo.

## 1. Dependencias  *(~2 min)*

In [ ]:
!pip -q install 'shapely>=2.0' geopandas pyogrio pyproj requests pandas numpy pyarrow
import shapely, geopandas, pandas as pd
print('shapely', shapely.__version__, '| geopandas', geopandas.__version__, '| pandas', pd.__version__)
assert int(shapely.__version__.split('.')[0]) >= 2, 'se necesita shapely 2.x'
import psutil
print(f'RAM total: {psutil.virtual_memory().total/1e9:.1f} GB')

## 2. Cargar el WDPA

Sube los 3 zips a `MyDrive/wdpa/` y monta el Drive.

In [ ]:
USE_DRIVE = True
WDPA_DIR  = '/content/drive/MyDrive/wdpa'

import os, glob
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('\ncontenido de', WDPA_DIR, ':')
    for f in sorted(glob.glob(f'{WDPA_DIR}/*')): print('  ', os.path.basename(f))
else:
    from google.colab import files
    up = files.upload(); WDPA_DIR = '.'
    print('subido:', list(up.keys()))

## 3. Token de GFW

In [ ]:
from getpass import getpass
GFW_TOKEN = getpass('Pega tu GFW API token (no se mostrara): ').strip()
print('token cargado:', bool(GFW_TOKEN), '| largo:', len(GFW_TOKEN))

## 4. Configuracion

`MPA_SELECT='reachable'` es el modo nacional. `MAX_MPAS=0` = todas las que pasen el filtro.
Con el filtro marino funcionando y la geometria domada, ya no hace falta tope.

In [ ]:
YEARS      = list(range(2019, 2026))
MPA_SELECT = 'reachable'      # 'pilot' | 'reachable' | 'all'
R_MAX_KM   = 1850.0
MAX_MPAS   = 0                # 0 = sin tope
OUT        = 'data_v6'

GFW_RES    = 'LOW'            # LOW = 0.1 grados (anida exacto en nuestras celdas). HIGH = 0.01 (100x mas datos)
BIN_EPS    = 1e-4             # empuje hacia el interior de la celda GFW al binear (evita borde por float)
LAND_MASK  = True             # descarta celdas cuyo centro cae en tierra (Natural Earth 50m)
N_WORKERS  = 1                # hilos de descarga
ROW_BATCH  = 400

# filtro marino
MIN_MARINE_AREA_KM2 = 25.0    # descarta PAs con componente marina despreciable
KEEP_STATUS = {'designated','inscribed','established'}   # excluye 'proposed'

# geometria
MAX_VERTICES = 20000
VERBOSE_GEOM = True

# reglas: (patron, buffer_km, delta, case). Primer match gana.
MPA_RULES = [
    ('rapa nui',      800.0, 0.2, 'oceanic'),
    ('motu motiro',   800.0, 0.2, 'oceanic'),
    ('salas y gomez', 800.0, 0.2, 'oceanic'),
    ('desventurad',   800.0, 0.2, 'oceanic'),
    ('nazca',         800.0, 0.2, 'oceanic'),
    ('juan fern',     400.0, 0.2, 'oceanic'),
]
OCEANIC_KM   = 500.0                        # a mas de esto del continente -> oceanica
RULE_COASTAL = (150.0, 0.1, 'coastal')
RULE_OCEANIC = (800.0, 0.2, 'oceanic')
SOUTH_LAT    = -40.0                        # coastal se subdivide en north/south para el reporte

PILOT_KEYS = ['diego ram', 'cabo de hornos', 'rapa nui', 'motu motiro']
MAX_CELLS_WARN = 400000

_zips = sorted(glob.glob(f'{WDPA_DIR}/WDPA_*shp_*.zip'))
_geo  = sorted(glob.glob(f'{WDPA_DIR}/*.geojson')) + sorted(glob.glob(f'{WDPA_DIR}/*.shp'))
WDPA_INPUT = _zips if _zips else (_geo[0] if _geo else 'WDPA_CHL.geojson')
ISO3 = 'CHL'
os.makedirs(f'{OUT}/raw', exist_ok=True)
print('WDPA_INPUT =', WDPA_INPUT)
print(f'{YEARS[0]}-{YEARS[-1]} | select={MPA_SELECT} | max_mpas={MAX_MPAS or "sin tope"} | res={GFW_RES} | workers={N_WORKERS}')

## 5. Bases y aeronaves

In [ ]:
import numpy as np, pandas as pd

BASES = pd.DataFrame([
    ('IQQ','Iquique',        -70.18,-20.53,'grupo',        5),
    ('TQM','Torquemada',     -71.48,-32.95,'casa',        99),
    ('THN','Talcahuano',     -73.10,-36.72,'grupo',        5),
    ('PMC','Puerto Montt',   -73.09,-41.44,'grupo',        5),
    ('PUQ','Punta Arenas',   -70.85,-53.00,'grupo',        5),
    ('PWM','Puerto Williams',-67.62,-54.93,'destacamento', 3),
    ('IPC','Mataveri',      -109.42,-27.16,'avanzada',     2),
], columns=['base_id','name','lon','lat','tier','sorties_wk'])

PLATFORMS = pd.DataFrame([
    ('P3',  'P-3ACH Orion',    555, 12.0, 1850, 40, 2),
    ('C295','C-295 Persuader', 480, 11.0, 1100, 30, 3),
], columns=['plat_id','name','v_kmh','T_h','R_km','W_km','n_avail'])

COMPAT = {'P3':   ['TQM','IPC','PUQ'],
          'C295': ['TQM','IQQ','THN','PMC','PUQ','PWM']}

MAINLAND_PORTS = pd.DataFrame([
    ('Arica',-70.32,-18.47),('Iquique',-70.15,-20.21),('Antofagasta',-70.40,-23.65),
    ('Caldera',-70.82,-27.07),('Coquimbo',-71.34,-29.95),('Valparaiso',-71.63,-33.04),
    ('San Antonio',-71.62,-33.59),('Talcahuano',-73.11,-36.71),('Valdivia',-73.40,-39.87),
    ('Puerto Montt',-72.94,-41.47),('Castro',-73.77,-42.48),('Puerto Chacabuco',-72.82,-45.46),
    ('Puerto Natales',-72.51,-51.73),('Punta Arenas',-70.91,-53.16),
    ('Puerto Williams',-67.62,-54.93)], columns=['port','lon','lat'])

print(BASES.to_string(index=False)); print()
print(PLATFORMS.to_string(index=False))

## 6. Helpers — solo ejecutar

In [ ]:
import json, time, sys, math, threading, warnings
import geopandas as gpd
import shapely
from shapely import contains_xy, prepare, points as shp_points
from shapely.geometry import Point, MultiPolygon
from shapely.ops import transform as shp_transform
from pyproj import CRS, Transformer

GFW_API = 'https://gateway.api.globalfishingwatch.org/v3'

# ---------------------------------------------------------------- geo basico
def haversine_km(lon1, lat1, lon2, lat2):
    R=6371.0; p1,p2=np.radians(lat1),np.radians(lat2)
    dphi=p2-p1; dl=np.radians(lon2-lon1)
    a=np.sin(dphi/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(np.clip(a,0,1)))

def cell_area_km2(lat, d):
    return (d*111.32)*(d*111.32*np.cos(np.radians(lat)))

def _aeqd(lat0, lon0):
    return CRS.from_proj4(f'+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m +no_defs')

def _fwd_inv(geom):
    c=geom.centroid; crs=_aeqd(c.y,c.x)
    return (Transformer.from_crs(4326,crs,always_xy=True),
            Transformer.from_crs(crs,4326,always_xy=True), crs)

def mpa_area_km2(geom):
    fwd,_,_=_fwd_inv(geom)
    return shp_transform(fwd.transform, geom).area/1e6

def n_coords(g):
    try: return int(shapely.get_num_coordinates(g))
    except Exception: return 0

def drop_small_parts(geom, min_area_deg2):
    if geom.geom_type != 'MultiPolygon': return geom
    parts=[p for p in geom.geoms if p.area >= min_area_deg2]
    if not parts: parts=[max(geom.geoms, key=lambda p: p.area)]
    return parts[0] if len(parts)==1 else MultiPolygon(parts)

def decimate(geom, max_pts, tol, tol_max):
    """Simplifica progresivamente hasta max_pts vertices, sin pasar de tol_max."""
    g=geom
    for _ in range(12):
        if n_coords(g) <= max_pts or tol > tol_max: break
        g2=g.simplify(tol, preserve_topology=True)
        if g2.is_empty or not g2.is_valid:
            g2=g.buffer(0).simplify(tol, preserve_topology=True)
        if n_coords(g2) < n_coords(g): g=g2
        tol*=2
    return g

def buffer_km_fast(geom, km, quad_segs=2):
    if km <= 0: return geom
    fwd,inv,_=_fwd_inv(geom)
    return shp_transform(inv.transform, shp_transform(fwd.transform, geom).buffer(km*1000.0, quad_segs=quad_segs))

_LAND=None
def load_land_mask():
    """Natural Earth land (50m), recortado a la zona de Chile, unido y preparado. None si falla."""
    global _LAND
    if _LAND is not None: return _LAND
    urls=['https://naciscdn.org/naturalearth/50m/physical/ne_50m_land.zip',
          'https://github.com/nvkelso/natural-earth-vector/raw/master/zips/50m_physical/ne_50m_land.zip',
          'https://naciscdn.org/naturalearth/110m/physical/ne_110m_land.zip']
    for u in urls:
        try:
            land=gpd.read_file(u).to_crs(4326)
            land=land.cx[-125:-60, -62:-15]
            geom=shapely.union_all(land.geometry.values)
            prepare(geom); _LAND=geom
            print(f'  mascara de tierra: {u.split("/")[-1]} ({n_coords(geom):,} vertices)')
            return _LAND
        except Exception as ex:
            print(f'  (no se pudo bajar {u.split("/")[-1]}: {str(ex)[:80]})')
    print('  *** sin mascara de tierra: las celdas costeras en tierra quedan en la grilla')
    return None

def bbox_expand_km(geom, km):
    """bbox de la geometria cruda + margen en km. Instantaneo, no bufferea nada."""
    minx,miny,maxx,maxy=geom.bounds
    dlat=km/111.32
    latm=max(abs(miny),abs(maxy))
    dlon=km/(111.32*max(np.cos(np.radians(latm)),0.05))
    x0,x1=max(minx-dlon,-180),min(maxx+dlon,180)
    y0,y1=max(miny-dlat,-90),min(maxy+dlat,90)
    return {'type':'Polygon','coordinates':[[[x0,y0],[x1,y0],[x1,y1],[x0,y1],[x0,y0]]]}

def classify_mpa(name, geom):
    n=str(name).lower()
    for pat,b,d,cs in MPA_RULES:
        if pat in n: return b,d,cs
    c=geom.centroid
    dmin=min(haversine_km(c.x,c.y,p.lon,p.lat) for p in MAINLAND_PORTS.itertuples())
    return RULE_OCEANIC if dmin>OCEANIC_KM else RULE_COASTAL

# ---------------------------------------------------------------- WDPA
def _col(g, *names):
    """Busca una columna sin distinguir mayusculas. Devuelve el nombre real o None."""
    low={c.lower():c for c in g.columns}
    for n in names:
        if n.lower() in low: return low[n.lower()]
    return None

def _read_one(shp, iso3):
    if iso3:
        try: return gpd.read_file(shp, where=f"ISO3 LIKE '%{iso3}%'", engine='pyogrio')
        except Exception: pass
    g=gpd.read_file(shp)
    c=_col(g,'ISO3')
    if iso3 and c: g=g[g[c].astype(str).str.contains(iso3, na=False)]
    return g

def load_mpas(path_or_zips, iso3=None):
    import zipfile, glob as _glob
    paths=path_or_zips if isinstance(path_or_zips,(list,tuple)) else [path_or_zips]
    frames=[]
    for p in paths:
        if str(p).lower().endswith('.zip'):
            d=str(p)[:-4]+'_ext'
            if not os.path.isdir(d):
                print(f'  descomprimiendo {os.path.basename(str(p))} ...')
                with zipfile.ZipFile(p) as z: z.extractall(d)
            shps=[x for x in _glob.glob(f'{d}/**/*.shp', recursive=True) if 'point' not in x.lower()]
            if not shps: shps=_glob.glob(f'{d}/**/*.shp', recursive=True)
        else: shps=[p]
        for shp in shps:
            print(f'  leyendo {os.path.basename(str(shp))} ...')
            gg=_read_one(shp, iso3)
            if len(gg): frames.append(gg)
    if not frames: raise RuntimeError('No se leyo ningun poligono.')
    g=pd.concat(frames, ignore_index=True)
    g=gpd.GeoDataFrame(g, geometry='geometry', crs=frames[0].crs).to_crs(4326)
    n0=len(g)
    print(f'\n  registros crudos: {n0}')
    print(f'  columnas: {list(g.columns)}')

    # ---- 1) tipo de area: solo PAs (PA_DEF=1), no OECMs --------------------
    c=_col(g,'PA_DEF')
    if c:
        pad=pd.to_numeric(g[c], errors='coerce')
        print(f'  PA_DEF: {pad.value_counts(dropna=False).to_dict()}')
        g=g[pad.fillna(1)==1]

    # ---- 2) marino: cascada MARINE -> area marina -> palabras clave ---------
    c=_col(g,'MARINE')
    if c:
        mar=pd.to_numeric(g[c], errors='coerce')
        print(f'  MARINE: {mar.value_counts(dropna=False).to_dict()}')
        g=g[mar.isin([1,2])]
        print(f'  tras MARINE in (1,2): {len(g)}')
        marine_filter='MARINE'
    else:
        ca=_col(g,'REP_M_AREA','GIS_M_AREA')
        if ca:
            am=pd.to_numeric(g[ca], errors='coerce').fillna(0)
            g=g[am>0]; marine_filter=ca
            print(f'  (sin MARINE) tras {ca}>0: {len(g)}')
        else:
            cd=_col(g,'DESIG_ENG','DESIG','NAME')
            kw='marin|marine|coastal|costero|mar |maritim'
            g=g[g[cd].astype(str).str.lower().str.contains(kw, na=False)]
            marine_filter=f'palabras clave en {cd}'
            print(f'  (sin MARINE ni area marina) tras palabras clave: {len(g)}')
    if len(g)==n0 and n0>60:
        raise RuntimeError(f'El filtro marino ({marine_filter}) no descarto nada ({n0}). '
                           f'Revisa las columnas impresas arriba y avisame.')

    # ---- 3) area marina minima ---------------------------------------------
    ca=_col(g,'REP_M_AREA','GIS_M_AREA')
    if ca:
        am=pd.to_numeric(g[ca], errors='coerce').fillna(0)
        cnm_=_col(g,'NAME')
        drop=g[am<MIN_MARINE_AREA_KM2]
        if len(drop) and cnm_:
            print(f'  descartadas por {ca} < {MIN_MARINE_AREA_KM2} km2:')
            for nm,a in zip(drop[cnm_].astype(str), am[am<MIN_MARINE_AREA_KM2]): print(f'      {nm[:45]:45s} {a:8.1f} km2')
        g=g[am>=MIN_MARINE_AREA_KM2]
        print(f'  tras {ca} >= {MIN_MARINE_AREA_KM2} km2: {len(g)}')

    # ---- 4) estado legal ---------------------------------------------------
    c=_col(g,'STATUS')
    if c:
        st=g[c].astype(str).str.strip().str.lower()
        print(f'  STATUS: {st.value_counts().to_dict()}')
        g=g[st.isin(KEEP_STATUS)]
        print(f'  tras STATUS in {sorted(KEEP_STATUS)}: {len(g)}')

    # ---- 5) columnas normalizadas ------------------------------------------
    cid=_col(g,'WDPAID','SITE_ID'); cnm=_col(g,'NAME'); cnt=_col(g,'NO_TAKE')
    g['mpa_id']  = g[cid].astype(str) if cid else g.index.astype(str)
    g['mpa_name']= g[cnm].astype(str) if cnm else g['mpa_id']
    if cnt:
        nt=g[cnt].astype(str).str.strip().str.lower()
        g['nt_ord']=np.select([nt.eq('all'), nt.eq('part')],[2,1],0)
    else:
        g['nt_ord']=0
    g=g[~g.geometry.is_empty & g.geometry.notna()]

    # ---- 6) parcelas de una misma AMP -> union -----------------------------
    dup=g['mpa_id'].duplicated(keep=False)
    if dup.any():
        n_multi=g.loc[dup,'mpa_id'].nunique()
        print(f'  {n_multi} AMPs con varias parcelas -> dissolve')
        gd=(g[dup][['mpa_id','mpa_name','nt_ord','geometry']]
              .dissolve(by='mpa_id', aggfunc={'mpa_name':'first','nt_ord':'max'}).reset_index())
        g=pd.concat([g[~dup][['mpa_id','mpa_name','nt_ord','geometry']], gd], ignore_index=True)
    else:
        g=g[['mpa_id','mpa_name','nt_ord','geometry']]
    g=gpd.GeoDataFrame(g, geometry='geometry', crs=4326)
    g['no_take']=(g.nt_ord>0).astype(int)
    g['no_take_kind']=g.nt_ord.map({0:'none',1:'part',2:'all'})
    print(f'  FILTRO FINAL: {n0} -> {len(g)} AMPs marinas vigentes')
    return g.drop(columns='nt_ord').reset_index(drop=True)

# ---------------------------------------------------------------- alcance (vectorizado)
def base_reach(lon, lat):
    lon=np.atleast_1d(lon).astype(float); lat=np.atleast_1d(lat).astype(float)
    n=len(lon)
    D=np.stack([haversine_km(lon,lat,b.lon,b.lat) for b in BASES.itertuples()],axis=1)
    plat_ids=list(PLATFORMS.plat_id); R=dict(zip(PLATFORMS.plat_id,PLATFORMS.R_km))
    M=np.zeros((n,len(plat_ids)),bool)                    # celda x plataforma: la cubre alguna base
    best_b=np.full(n,'---',dtype=object); best_d=np.full(n,np.inf)
    for j,b in enumerate(BASES.itertuples()):
        ok=[p for p in plat_ids if b.base_id in COMPAT.get(p,[])]
        if not ok: continue
        rmax=max(R[p] for p in ok)
        reach=D[:,j]<=rmax
        upd=reach&(D[:,j]<best_d)
        best_b[upd]=b.base_id; best_d[upd]=D[upd,j]
        for pi,p in enumerate(plat_ids):
            if p in ok: M[:,pi]|=D[:,j]<=R[p]
    codes=M@(1<<np.arange(len(plat_ids)))
    lut=np.array(['|'.join(p for pi,p in enumerate(plat_ids) if (c>>pi)&1) for c in range(1<<len(plat_ids))],dtype=object)
    return best_b, best_d, lut[codes]

# ---------------------------------------------------------------- grilla
def signed_dist_km(core, lo, la, inside):
    """Distancia (km) de cada punto al borde de core; negativa si inside. STRtree sobre segmentos."""
    fwd,_,crs=_fwd_inv(core)
    xs,ys=fwd.transform(lo,la)
    bnd=shp_transform(fwd.transform, core.boundary)
    segs=[]
    for part in shapely.get_parts(bnd):
        cc=shapely.get_coordinates(part)
        if len(cc)>=2: segs.append(shapely.linestrings(np.stack([cc[:-1],cc[1:]],axis=1)))
    if not segs: return np.zeros(len(lo))
    segs=np.concatenate(segs)
    tree=shapely.STRtree(segs)
    pts=shp_points(xs,ys)
    idx=tree.nearest(pts)
    d=shapely.distance(pts, segs[idx])/1000.0
    return np.where(inside,-d,d)

def cells_for_mpa(m, row_batch=ROW_BATCH):
    bkm, delta, case = classify_mpa(m.mpa_name, m.geometry)
    raw=m.geometry; n0=n_coords(raw)
    core=drop_small_parts(raw, min_area_deg2=(delta*delta)/4.0)
    core=decimate(core, MAX_VERTICES, tol=delta/10.0, tol_max=delta/2.0)
    if core.is_empty: return None
    area=mpa_area_km2(core)
    coarse=decimate(core, MAX_VERTICES//5, tol=delta/2.0, tol_max=delta*2)
    ext=buffer_km_fast(coarse, bkm, quad_segs=2)
    ext=decimate(ext, MAX_VERTICES, tol=delta/2.0, tol_max=delta*2)
    if VERBOSE_GEOM:
        print(f'      vertices {n0:>7d} -> core {n_coords(core):>6d} / ext {n_coords(ext):>6d}')
    prepare(core); prepare(ext)
    minx,miny,maxx,maxy=ext.bounds
    lons=np.arange(np.floor(minx/delta)*delta, maxx+delta, delta)+delta/2
    lats=np.arange(np.floor(miny/delta)*delta, maxy+delta, delta)+delta/2
    K_lon=[]; K_lat=[]; K_in=[]
    for i0 in range(0,len(lats),row_batch):
        LON,LAT=np.meshgrid(lons, lats[i0:i0+row_batch])
        lo=LON.ravel(); la=LAT.ravel()
        sel=contains_xy(ext, lo, la)
        if not sel.any(): continue
        lo=lo[sel]; la=la[sel]
        K_lon.append(lo); K_lat.append(la); K_in.append(contains_xy(core, lo, la))
    if not K_lon: return None
    lo=np.concatenate(K_lon); la=np.concatenate(K_lat); inside=np.concatenate(K_in)
    d=signed_dist_km(core, lo, la, inside)
    return pd.DataFrame(dict(
        lon=np.round(lo,6), lat=np.round(la,6),
        zone=np.where(inside,'interior','ring'), dist_bnd_km=d,
        mpa_id=m.mpa_id, mpa_name=m.mpa_name,
        no_take=np.where(inside, int(m.no_take), 0),
        no_take_kind=np.where(inside, m.no_take_kind, 'none'),
        delta=delta, buffer_km=bkm, case=case, mpa_area_km2=area))

def build_cells(mpas):
    out=[]; t0=time.time()
    for k,m in enumerate(mpas.itertuples(),1):
        t1=time.time(); df=cells_for_mpa(m)
        n=0 if df is None else len(df)
        if n: out.append(df)
        print(f'  [{k}/{len(mpas)}] {str(m.mpa_name)[:34]:34s} {(df.case.iloc[0] if n else "-"):8s} -> {n:7d} celdas  ({time.time()-t1:5.1f}s)')
    if not out: return pd.DataFrame()
    c=pd.concat(out, ignore_index=True); n0=len(c)
    # dedup global: interior > ring; interior -> AMP mas chica; ring -> borde mas cercano
    grp=c.groupby(['lon','lat'])
    no_take_any=grp.no_take.max(); n_desig=grp.size()
    _ord=c.no_take_kind.map({'none':0,'part':1,'all':2}).fillna(0).astype(int)
    ntk=_ord.groupby([c.lon,c.lat]).max().map({0:'none',1:'part',2:'all'})
    c['_r']=(c.zone!='interior').astype(int)
    c['_k']=np.where(c.zone.values=='interior', c.mpa_area_km2.values, c.dist_bnd_km.values)
    c=c.sort_values(['_r','_k']).drop_duplicates(subset=['lon','lat']).drop(columns=['_r','_k']).reset_index(drop=True)
    key=pd.MultiIndex.from_arrays([c.lon,c.lat])
    c['no_take']=no_take_any.reindex(key).values
    c['no_take_kind']=ntk.reindex(key).values
    c['n_desig']=n_desig.reindex(key).values
    print(f'\n  dedup global: {n0} -> {len(c)} celdas ({n0-len(c)} duplicadas)')
    # marca celdas gruesas que se solapan con celdas finas (solo informativo)
    deltas=sorted(c.delta.unique())
    c['overlap_fine']=0
    if len(deltas)>1:
        fine=c[c.delta==deltas[0]]; fkey=set(zip(fine.lon.round(6),fine.lat.round(6)))
        d0=deltas[0]
        for d in deltas[1:]:
            sub=c.delta==d
            cx=c.loc[sub,'lon'].values; cy=c.loc[sub,'lat'].values
            hit=np.zeros(sub.sum(),bool)
            for ox in np.arange(-d/2+d0/2, d/2, d0):
                for oy in np.arange(-d/2+d0/2, d/2, d0):
                    kk=zip(np.round(cx+ox,6),np.round(cy+oy,6))
                    hit|=np.fromiter((k_ in fkey for k_ in kk),bool,len(cx))
            c.loc[sub,'overlap_fine']=hit.astype(int)
        print(f'  celdas gruesas solapadas con finas: {int(c.overlap_fine.sum())} (informativo)')
    # celdas en tierra: no son objetivo de patrulla ni pueden tener esfuerzo
    if LAND_MASK:
        land=load_land_mask()
        if land is not None:
            onland=contains_xy(land, c.lon.values, c.lat.values)
            print(f'  celdas en tierra descartadas: {int(onland.sum())} de {len(c)}')
            c=c[~onland].reset_index(drop=True)
    c['cell_id']=np.arange(len(c))
    c['area_km2']=cell_area_km2(c.lat.values, c.delta.values)
    c['region']=np.where(c.case=='oceanic','oceanic',np.where(c.lat<SOUTH_LAT,'coastal_south','coastal_north'))
    b,d,p=base_reach(c.lon.values, c.lat.values)
    c['sector']=b; c['dist_base_km']=d; c['platforms_any']=p
    c['reachable']=(c.sector!='---').astype(int)
    print(f'  total: {time.time()-t0:.1f}s')
    return c

# ---------------------------------------------------------------- GFW
def fetch_effort(token, geometry, d0, d1, timeout=600, attempts=4):
    import requests
    url=(f'{GFW_API}/4wings/report?spatial-resolution={GFW_RES}&temporal-resolution=MONTHLY'
         f'&group-by=GEARTYPE&format=JSON'
         f'&datasets[0]=public-global-fishing-effort:v4.0&date-range={d0},{d1}')
    last=''
    for a in range(attempts):
        try:
            r=requests.post(url, headers={'Authorization':f'Bearer {token}','Content-Type':'application/json'},
                            data=json.dumps({'geojson':geometry}), timeout=timeout)
            if r.status_code==200: return r.json()
            last=f'GFW API {r.status_code}: {r.text[:300]}'
            if r.status_code not in (429,500,502,503,504): raise RuntimeError(last)
        except requests.RequestException as ex:
            last=f'red: {ex}'
        time.sleep(5*(2**a))
    raise RuntimeError(f'agotados {attempts} intentos -- {last}')

LATK=('lat','latitude','cell_ll_lat'); LONK=('lon','long','longitude','cell_ll_lon')
VALK=('hours','value','fishing_hours','detections','count')

def flatten_report(js):
    recs=[]
    def has(keys,c): return any(k==x or k.endswith('_'+x) for k in keys for x in c)
    def walk(n):
        if isinstance(n,dict):
            ks={k.lower() for k in n}
            if has(ks,LATK) and has(ks,LONK) and has(ks,VALK): recs.append(n)
            else:
                for v in n.values(): walk(v)
        elif isinstance(n,list):
            for v in n: walk(v)
    walk(js)
    if not recs: return pd.DataFrame()
    df=pd.DataFrame(recs); ren={}
    for c in df.columns:
        cl=c.lower()
        if cl in LATK: ren[c]='lat'
        elif cl in LONK: ren[c]='lon'
        elif cl in VALK: ren[c]='hours'
        elif cl in ('date','time','timestamp'): ren[c]='date'
        elif 'gear' in cl: ren[c]='geartype'
        elif 'vessel' in cl: ren[c]='n_vessels'
    return df.rename(columns=ren)

def bin_to_cells(e, cells_df):
    """Asigna cada registro GFW a UNA celda. Si hay celdas de distinto delta en el mismo lugar,
    gana la mas fina. Sin doble conteo."""
    e=e.reset_index(drop=True); e['rid']=np.arange(len(e))
    if 'gear' not in e.columns: e['gear']='other'
    if 'n_vessels' not in e.columns: e['n_vessels']=0
    parts=[]
    for d in sorted(cells_df.delta.unique()):
        cd=cells_df.loc[cells_df.delta==d, ['cell_id','lon','lat','no_take','zone']]
        t=e[['rid','date','hours','n_vessels','gear','lon','lat']].copy()
        # +EPS empuja el punto hacia el interior de su celda GFW: un registro exactamente en
        # el borde de un bin no cae al vecino por error de punto flotante
        t['lon']=(np.floor((t.lon+BIN_EPS)/d)*d+d/2).round(6); t['lat']=(np.floor((t.lat+BIN_EPS)/d)*d+d/2).round(6)
        parts.append(t.merge(cd, on=['lon','lat'], how='inner'))
    if not parts: return pd.DataFrame()
    return pd.concat(parts, ignore_index=True).drop_duplicates('rid', keep='first')

print('helpers listos')

## 6b. Autotest — verifica la logica antes de tocar la API

Si algo aqui dice `FAIL`, **para y avisame**. No sigas a la descarga.

In [ ]:
from shapely.geometry import Polygon, box
_ok=True
def chk(name, cond):
    global _ok
    print(('  OK   ' if cond else '  FAIL ')+name); _ok&=bool(cond)

# 1) alineacion de grilla: arange+delta/2 == floor-binning, para 0.1 y 0.2
for d in (0.1, 0.2):
    rng=np.random.default_rng(0); x=rng.uniform(-110,-66,20000)
    a=(np.floor(x/d)*d+d/2).round(6)
    lo=np.floor(x.min()/d)*d; grid=np.round(np.arange(lo, x.max()+d, d)+d/2, 6)
    chk(f'binning alineado con la grilla (delta={d})', np.isin(a, grid).all())
# 2) anidamiento GFW LOW (0.1) en 0.2: centros y esquinas de celdas 0.1 caen en el mismo bin 0.2
g01=np.arange(-109.95,-100,0.1)                  # centros de celdas GFW 0.1
ll=g01-0.05                                       # esquinas inferiores
chk('GFW 0.1 anida en 0.2 (centro y esquina al mismo bin)',
    np.allclose(np.floor(g01/0.2), np.floor((ll+1e-9)/0.2)))
# 2c) EPS: un registro GFW exactamente en un borde de bin 0.2 va al bin correcto
x=np.array([-109.8, -69.6, -55.0])             # multiplos exactos de 0.2 (borde de bin)
b_eps=np.floor((x+BIN_EPS)/0.2)*0.2+0.1
chk('registro en borde exacto de bin -> bin de la derecha (su propia celda)', np.allclose(b_eps, x+0.1))
# 3) contains_xy con geometria preparada
sq=box(-70,-56,-68,-54); prepare(sq)
chk('contains_xy prepared', bool(contains_xy(sq,-69,-55)) and not bool(contains_xy(sq,-60,-55)))
# 4) distancia con signo vs formula directa en un cuadrado
lo=np.array([-69.0,-67.0]); la=np.array([-55.0,-55.0]); inside=np.array([True,False])
d=signed_dist_km(sq, lo, la, inside)
chk('distancia con signo (interior negativa, exterior positiva)', d[0]<0 and d[1]>0)
chk('distancia exterior ~64 km (1 grado lon a 55S)', abs(d[1]-63.9)<3)
# 5) alcance
b,dd,p=base_reach(np.array([-68.72,-109.35,-80.10]), np.array([-56.52,-27.12,-26.30]))
chk('Diego Ramirez -> PWM', b[0]=='PWM'); chk('Rapa Nui -> IPC solo P3', b[1]=='IPC' and p[1]=='P3')
chk('Nazca-Desventuradas -> P3 llega, C295 no', 'P3' in p[2] and 'C295' not in p[2])
# 6) dedup: interior gana a ring; interior mas chica; ring mas cercana
t=pd.DataFrame([dict(lon=1.,lat=1.,zone='interior',dist_bnd_km=-30.,mpa_area_km2=7e5,no_take=0,no_take_kind='none',mpa_id='A',mpa_name='A',delta=0.2,buffer_km=800,case='oceanic'),
                dict(lon=1.,lat=1.,zone='interior',dist_bnd_km=-90.,mpa_area_km2=1.5e5,no_take=1,no_take_kind='all',mpa_id='B',mpa_name='B',delta=0.2,buffer_km=800,case='oceanic'),
                dict(lon=2.,lat=2.,zone='ring',dist_bnd_km=300.,mpa_area_km2=7e5,no_take=0,no_take_kind='none',mpa_id='A',mpa_name='A',delta=0.2,buffer_km=800,case='oceanic'),
                dict(lon=2.,lat=2.,zone='ring',dist_bnd_km=120.,mpa_area_km2=1.5e5,no_take=0,no_take_kind='none',mpa_id='B',mpa_name='B',delta=0.2,buffer_km=800,case='oceanic')])
grp=t.groupby(['lon','lat']); nta=grp.no_take.max()
t['_r']=(t.zone!='interior').astype(int); t['_k']=np.where(t.zone=='interior',t.mpa_area_km2,t.dist_bnd_km)
t=t.sort_values(['_r','_k']).drop_duplicates(['lon','lat']).set_index(['lon','lat'])
chk('dedup interior -> AMP mas chica', t.loc[(1.,1.),'mpa_id']=='B')
chk('dedup ring -> AMP mas cercana',   t.loc[(2.,2.),'mpa_id']=='B')
chk('no_take se propaga (max)',        nta.loc[(1.,1.)]==1)
# 7) targets y binning finer-first sin doble conteo
cells_t=pd.DataFrame([dict(cell_id=0,lon=-69.55,lat=-55.05,no_take=1,zone='interior',delta=0.1),
                      dict(cell_id=1,lon=-69.5, lat=-55.1, no_take=0,zone='ring',    delta=0.2)])
e_t=pd.DataFrame([dict(date='2020-01',hours=10.,n_vessels=1,gear='trawl',lon=-69.52,lat=-55.03),   # cae en ambas
                  dict(date='2020-01',hours=7., n_vessels=1,gear='squid',lon=-69.45,lat=-55.15),   # solo en la gruesa
                  dict(date='2020-01',hours=3., n_vessels=1,gear='other',lon=-69.6, lat=-55.2)])   # borde exacto de bin
mg=bin_to_cells(e_t, cells_t)
chk('registro en 2 celdas de distinto delta -> 1 sola (la fina)', (mg.rid==0).sum()==1 and mg[mg.rid==0].cell_id.iloc[0]==0)
chk('registro solo cubierto por la gruesa -> va a la gruesa', (mg.rid==1).sum()==1 and mg[mg.rid==1].cell_id.iloc[0]==1)
chk('registro en borde exacto (-69.6,-55.2) -> celda gruesa', (mg.rid==2).sum()==1 and mg[mg.rid==2].cell_id.iloc[0]==1)
chk('sin doble conteo (10+7+3=20)', mg.hours.sum()==20.0 and len(mg)==3)
chk('gear se conserva', set(mg.gear)=={'trawl','squid','other'})
interior=(mg.zone=='interior').values
yv=np.where(interior&(mg.no_take==1).values, mg.hours, 0).sum(); yp=np.where(~interior, mg.hours, 0).sum()
chk('y_viol=10, y_press=10', yv==10 and yp==10)
# 8) bin_to_cells sin columna gear tampoco revienta
mg2=bin_to_cells(e_t.drop(columns=['gear','n_vessels']), cells_t)
chk('bin_to_cells tolera ausencia de gear/n_vessels', len(mg2)==3 and (mg2.gear=='other').all())
print('\nAUTOTEST', 'OK -- puedes seguir' if _ok else 'FALLO -- no sigas, avisame')
assert _ok

## 7. Seleccionar AMPs y construir la grilla

**Revisa la lista de nombres que imprime.** Tu conoces las AMPs: si ves un parque
terrestre (Torres del Paine, Lauca, etc.), el filtro sigue mal y me avisas.

In [ ]:
mpas_all = load_mpas(WDPA_INPUT, iso3=ISO3)
N_ALL = len(mpas_all)

if MPA_SELECT == 'pilot':
    pat='|'.join(PILOT_KEYS)
    mpas=mpas_all[mpas_all.mpa_name.str.lower().str.contains(pat,na=False)].reset_index(drop=True)
elif MPA_SELECT == 'reachable':
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        cen=mpas_all.geometry.representative_point()
    b,d,p=base_reach(cen.x.values, cen.y.values)
    mpas=mpas_all[b!='---'].reset_index(drop=True)
    print(f'\n  alcanzables: {len(mpas)}/{N_ALL}')
else:
    mpas=mpas_all.reset_index(drop=True)

if MAX_MPAS and len(mpas) > MAX_MPAS:
    mpas=(mpas.assign(_a=[mpa_area_km2(g) for g in mpas.geometry])
              .sort_values('_a', ascending=False).head(MAX_MPAS).drop(columns='_a').reset_index(drop=True))
    print(f'  tope aplicado -> {len(mpas)} AMPs')

# ---- LISTA PARA REVISION HUMANA ----
info=[]
for m in mpas.itertuples():
    bkm,dl,cs=classify_mpa(m.mpa_name,m.geometry)
    info.append(dict(mpa_id=m.mpa_id, nombre=m.mpa_name[:48], case=cs, no_take=m.no_take_kind, buffer=bkm, delta=dl))
info=pd.DataFrame(info).sort_values(['case','nombre'])
print(f'\n{len(mpas)} AMPs seleccionadas:\n')
print(info.to_string(index=False))
print('\n>>> Si aqui aparece algun parque TERRESTRE, para y avisame. <<<\n')

cells_df = build_cells(mpas)
cells_df.to_csv(f'{OUT}/cells.csv', index=False)
print(f'\nTOTAL celdas: {len(cells_df)}  (interior={int((cells_df.zone=="interior").sum())}, ring={int((cells_df.zone=="ring").sum())})')
print(cells_df.groupby('region').agg(celdas=('cell_id','size'), amps=('mpa_id','nunique'),
                                     alcanzables=('reachable','sum')).to_string())
if len(cells_df) > MAX_CELLS_WARN:
    print(f'\n*** GRILLA GRANDE ({len(cells_df)}). Sube los delta o pon MAX_MPAS.')
display(cells_df.head())

## 8. Descargar el esfuerzo pesquero — en paralelo

Resolucion `LOW` (0,1°): ~100x menos datos que antes, sin perder nada para nuestra grilla.
La bbox se calcula expandiendo los bounds (instantaneo). Cache en `data_v6/raw/`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

jobs=[]
for m in mpas.itertuples():
    bkm,_,_=classify_mpa(m.mpa_name, m.geometry)
    geom=bbox_expand_km(m.geometry, bkm*1.05)          # 5% de margen sobre el anillo
    for y in YEARS: jobs.append((m.mpa_id, str(m.mpa_name), geom, y))

lock=threading.Lock(); done=[0]; t0=time.time()
frames=[]; errors=[]; schema=[None]

def work(job):
    mid,mname,geom,y=job
    cache=f'{OUT}/raw/{mid}_{y}_{GFW_RES}.json'
    try:
        if os.path.exists(cache):
            js=json.load(open(cache))
        else:
            js=fetch_effort(GFW_TOKEN, geom, f'{y}-01-01', f'{y}-12-31')
            with open(cache,'w') as fh: json.dump(js, fh)
        df=flatten_report(js)
        if not df.empty: df['mpa_id']=mid
        return ('ok',mid,mname,y,df,js)
    except Exception as ex:
        return ('err',mid,mname,y,None,str(ex))

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futs=[ex.submit(work,j) for j in jobs]
    for f in as_completed(futs):
        st,mid,mname,y,df,extra=f.result()
        with lock:
            done[0]+=1; k=done[0]
            if st=='ok':
                if schema[0] is None:
                    schema[0]=extra
                    with open(f'{OUT}/SCHEMA_SAMPLE.txt','w') as fh:
                        fh.write(f'RESOLUTION: {GFW_RES}\nTOP-LEVEL KEYS: {list(extra.keys())[:20]}\n\n')
                        fh.write('RAW SNIPPET (2500 chars):\n'+json.dumps(extra)[:2500]+'\n')
                if df is not None and not df.empty: frames.append(df)
                n=0 if df is None else len(df)
            else:
                errors.append(f'{mname} {y}: {extra}'); n=-1
            el=time.time()-t0; eta=el/k*(len(jobs)-k)
            if k%5==0 or k==len(jobs) or n<0:
                print(f'  [{k}/{len(jobs)}] {mname[:30]:30s} {y}  {"ERR" if n<0 else f"filas={n:6d}"}  ETA {eta/60:5.1f} min')

print(f'\nbloques con datos: {len(frames)} | errores: {len(errors)} | {(time.time()-t0)/60:.1f} min')
if errors:
    print('primeros errores:'); [print('  ',e_[:160]) for e_ in errors[:5]]
    print('re-corre esta celda: lo cacheado no se vuelve a pedir.')

## 9. Panel mensual (sparse) — `y_viol` / `y_press` / `y_inside`

- **`y_viol`** — esfuerzo dentro de zona no-take. La infraccion.
- **`y_inside`** — todo el esfuerzo dentro del AMP (incluye uso multiple, donde puede ser legal).
- **`y_press`** — esfuerzo en el anillo. La presion sobre el limite.

Se guarda **sparse** (solo celda-mes con datos) en parquet. El panel denso se arma
en la etapa de modelamiento, por caso, cuando haga falta.

In [ ]:
N_MONTHS=len(YEARS)*12
if not frames:
    print('SIN DATOS PARSEADOS. Mandame SCHEMA_SAMPLE.txt.'); obs=pd.DataFrame()
else:
    effort=pd.concat(frames, ignore_index=True)
    with open(f'{OUT}/SCHEMA_SAMPLE.txt','a') as fh:
        fh.write('\n\nPARSED COLUMNS: '+str(list(effort.columns))+'\n')
        fh.write('FIRST 5 PARSED ROWS:\n'+effort.head().to_string()+'\n')
    e=effort.copy()
    e['date']=pd.to_datetime(e['date'], errors='coerce'); e=e.dropna(subset=['date'])
    if 'n_vessels' not in e.columns: e['n_vessels']=0
    # arte de pesca -> 4 buckets (util como feature del modelo de riesgo)
    g_=e['geartype'].astype(str).str.lower() if 'geartype' in e.columns else pd.Series('',index=e.index)
    e['gear']=np.select([g_.str.contains('trawl'), g_.str.contains('longline'),
                         g_.str.contains('squid|jigger'), g_.str.contains('seine')],
                        ['trawl','longline','squid','seine'], 'other')
    n_raw=len(e)
    gcol='geartype' if 'geartype' in e.columns else None
    e=e.drop_duplicates(subset=['date','lat','lon']+([gcol] if gcol else []))
    print(f'dedup esfuerzo crudo: {n_raw:,} -> {len(e):,} ({n_raw-len(e):,} solapadas)')
    # chequeo de resolucion real de GFW
    frac=np.abs((e.lon*100)%10); res_ok=(frac<1e-6)|(np.abs(frac-5)<1e-6)|(np.abs(frac-10)<1e-6)
    print(f'coordenadas GFW en multiplos de 0.1 (o centros 0.05): {100*res_ok.mean():.1f} %')
    mg=bin_to_cells(e, cells_df)
    print(f'registros asignados a celdas: {len(mg):,} / {len(e):,}')
    mg['year']=mg.date.dt.year; mg['month']=mg.date.dt.month
    interior=(mg.zone=='interior').values
    mg['y_viol']  =np.where(interior&(mg.no_take==1).values, mg.hours.values, 0.0)
    mg['y_inside']=np.where(interior, mg.hours.values, 0.0)
    mg['y_press'] =np.where(~interior, mg.hours.values, 0.0)
    for gk in ('trawl','longline','squid','seine','other'):
        mg[f'h_{gk}']=np.where(mg.gear.values==gk, mg.hours.values, 0.0)
    obs=(mg.groupby(['cell_id','year','month'], as_index=False)
           .agg(y_viol=('y_viol','sum'), y_inside=('y_inside','sum'), y_press=('y_press','sum'),
                all_hours=('hours','sum'), n_vessels=('n_vessels','sum'),
                h_trawl=('h_trawl','sum'), h_longline=('h_longline','sum'), h_squid=('h_squid','sum'),
                h_seine=('h_seine','sum'), h_other=('h_other','sum')))
    try: obs.to_parquet(f'{OUT}/monthly_sparse.parquet', index=False)
    except Exception: obs.to_csv(f'{OUT}/monthly_sparse.csv', index=False)
    print(f'celda-mes con datos : {len(obs):,}   (panel denso seria {len(cells_df)*N_MONTHS:,})')
    print(f'y_viol   : {obs.y_viol.sum():14,.0f} h')
    print(f'y_inside : {obs.y_inside.sum():14,.0f} h')
    print(f'y_press  : {obs.y_press.sum():14,.0f} h')

## 10. DIAGNOSTICO

In [ ]:
L=[]; A=L.append
A('='*78); A('DIAGNOSTIC v6 -- risk-based MPA surveillance (Paper 1)'); A('='*78)
A(f'MPAs seleccionadas       : {len(mpas)}  (de {N_ALL} marinas vigentes tras filtro)')
A(f'Modo / resolucion GFW    : {MPA_SELECT} / {GFW_RES}   max_mpas={MAX_MPAS or 0}')
A(f'Grid cells (post-dedup)  : {len(cells_df)}')
if len(cells_df):
    A(f'  interior               : {int((cells_df.zone=="interior").sum())}')
    A(f'  anillo                 : {int((cells_df.zone=="ring").sum())}')
    A(f'  no-take (solo interior): {int(cells_df.no_take.sum())}   kinds: {cells_df[cells_df.zone=="interior"].no_take_kind.value_counts().to_dict()}')
    A(f'  alcanzables            : {int(cells_df.reachable.sum())} / {len(cells_df)}')
    A(f'  con >1 designacion     : {int((cells_df.n_desig>1).sum())}')
    A(f'  gruesas solapadas      : {int(cells_df.overlap_fine.sum())}')
    A(''); A('  por region:')
    A('  '+cells_df.groupby('region').agg(celdas=('cell_id','size'), amps=('mpa_id','nunique'),
        interior=('zone', lambda s:(s=='interior').sum()), alcanzables=('reachable','sum')).to_string().replace('\n','\n  '))
A(f'Years                    : {YEARS[0]}-{YEARS[-1]}  ({N_MONTHS} meses)')
A(f'Download errors          : {len(errors)}')
A('')
if len(obs):
    cinfo=cells_df[['cell_id','zone','case','region','dist_bnd_km','mpa_name','sector','reachable']]
    mc=obs.merge(cinfo, on='cell_id')
    n_cells=len(cells_df); panel=n_cells*N_MONTHS
    A('--- TARGETS ---')
    for col in ('y_viol','y_inside','y_press','all_hours'): A(f'{col:10s}: {obs[col].sum():15,.0f} h')
    A('')
    A('--- SPARSITY (denominador = panel completo) ---')
    for col in ('y_viol','y_press'):
        nz=int((obs[col]>0).sum()); ev=int((obs.groupby('cell_id')[col].max()>0).sum())
        A(f'{col:8s}: {nz:9,} celda-mes >0 ({100*nz/max(panel,1):7.4f} %) | {ev:7,}/{n_cells:,} celdas ({100*ev/max(n_cells,1):5.2f} %)')
    A('')
    A('--- PERFIL FISHING-THE-LINE por REGION (h/celda; denominador = TODAS las celdas de la banda) ---')
    A('(negativo = dentro del AMP, positivo = fuera)')
    bins=[-1e9,-200,-100,-50,-20,0,20,50,100,200,400,800,1e9]
    lab=['<-200','-200/-100','-100/-50','-50/-20','-20/0','0/20','20/50','50/100','100/200','200/400','400/800','>800']
    cells_df['band']=pd.cut(cells_df.dist_bnd_km, bins=bins, labels=lab)
    mc['band']=pd.cut(mc.dist_bnd_km, bins=bins, labels=lab)
    for rg in sorted(cells_df.region.unique()):
        nc=cells_df[cells_df.region==rg].groupby('band', observed=False).size().rename('cells')
        hh=mc[mc.region==rg].groupby('band', observed=False).all_hours.sum().rename('h')
        bt=pd.concat([hh,nc],axis=1).fillna(0)
        bt['h_per_cell']=(bt.h/bt.cells.replace(0,np.nan)).round(2)
        A(f'  [{rg}]  ({int(nc.sum()):,} celdas, {cells_df[cells_df.region==rg].mpa_name.nunique()} AMPs)')
        A('  '+bt.to_string().replace('\n','\n  '))
        try:
            i_=bt.loc['-20/0','h_per_cell']; o_=bt.loc['0/20','h_per_cell']
            if i_>0: A(f'  --> salto en el limite: {o_/i_:.1f}x  ({i_:.2f} -> {o_:.2f})')
        except Exception: pass
        A(f'  --> pico en la banda {bt.h_per_cell.idxmax() if bt.h_per_cell.notna().any() else None}')
        A('')
    A('--- ESFUERZO POR ARTE (all_hours) ---')
    gs=obs[['h_trawl','h_longline','h_squid','h_seine','h_other']].sum()
    A('  '+' | '.join(f'{k[2:]}={v:,.0f}' for k,v in gs.items()))
    A('')
    A('--- ESFUERZO POR ANIO ---')
    A(obs.groupby('year').agg(y_viol=('y_viol','sum'), y_press=('y_press','sum'), celdas_activas=('cell_id','nunique')).to_string())
    A('')
    A('--- TOP 20 AMPs por INFRACCION (y_viol) ---')
    top=mc.groupby(['region','mpa_name']).agg(y_viol=('y_viol','sum'), y_inside=('y_inside','sum'), y_press=('y_press','sum')).sort_values('y_viol',ascending=False)
    A(top.head(20).to_string())
    A('')
    A('--- TOP 15 AMPs por PRESION (y_press) ---')
    A(top.sort_values('y_press',ascending=False).head(15).to_string())
    A('')
    A('--- ESFUERZO POR SECTOR (base alcanzable mas cercana) ---')
    sec=mc.groupby('sector').agg(y_viol=('y_viol','sum'), y_press=('y_press','sum'), amps=('mpa_name','nunique'))
    sec['cells']=cells_df.groupby('sector').size()
    A(sec.to_string())
    A('')
    A('--- NO ALCANZABLE (queda fuera del ruteo) ---')
    ur=mc[mc.reachable==0]
    A(f'celdas: {int((cells_df.reachable==0).sum()):,}   y_press: {ur.y_press.sum():,.0f} h   y_viol: {ur.y_viol.sum():,.0f} h')
if errors:
    A(''); A('ERRORS (primeros 5):'); [A('  '+str(e_)[:150]) for e_ in errors[:5]]
A('='*78)
txt='\n'.join(L); print(txt)
open(f'{OUT}/DIAGNOSTIC.txt','w').write(txt)

## 11. Bajar los dos archivos de texto

In [ ]:
try:
    from google.colab import files
    files.download(f'{OUT}/DIAGNOSTIC.txt'); files.download(f'{OUT}/SCHEMA_SAMPLE.txt')
except Exception as e:
    print('descarga manual:', e)

In [ ]:
from google.colab import drive
import os, shutil
drive.mount('/content/drive')

DEST = '/content/drive/MyDrive/acanav_paper1'
os.makedirs(DEST, exist_ok=True)

for f in ['cells.csv', 'monthly_sparse.parquet', 'DIAGNOSTIC.txt', 'SCHEMA_SAMPLE.txt']:
    src = f'{OUT}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DEST}/{f}')
        print(f'{f:24s} {os.path.getsize(src)/1e6:8.1f} MB  -> guardado')
    else:
        print(f'{f:24s} NO EXISTE')

print('\ncarpeta destino:', DEST)

In [ ]:
shutil.make_archive(f'{DEST}/raw_gfw_v6', 'zip', f'{OUT}/raw')
print('raw comprimido y guardado')